In [1]:
# **CommaSeparatedListOutputParser列表输出解析器示例**

In [3]:
from langchain_openai import ChatOpenAI
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.prompts import PromptTemplate
#构造列表解析器
output_parser = CommaSeparatedListOutputParser()
#返回解析器的解析格式
output_parser.get_format_instructions()

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [5]:
# 注意：所有解析器的解析格式都是英文的，上述列表解析器解析格式的英文翻译是：`您的响应应该是逗号分隔的值列表，
# 例如：`foo，bar，baz`或`foo，bar，baz`。也就是通过解析器的该种解析格式作为提示词的部分内容，约束模型按照指定格式进行内容的输出。

In [6]:
# 解析器作用在PromptTemplate模版中

In [8]:
#构造输入模版，这里的区别是：在输入的Prompt Template中，加入了OutPut Parse的内容
template = """用户发起的提问:

{question}

{format_instructions}"""

#实例化输出解析器（用于解析以逗号分隔的列表类型的输出）
output_parser = CommaSeparatedListOutputParser()

#创建提示词模版，将输出解析器的解析格式作为提示词模版的部分内容
prompt = PromptTemplate.from_template(
    template,
    partial_variables={"format_instructions":
                       output_parser.get_format_instructions()},
)


#最后，使用LangChain中的`chain`的抽象，合并最终的提示、大模型实例及OutPut Parse共同执行。
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()

model = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")

chain = prompt | model | output_parser
output = chain.invoke({"question": "列出北京的三个景点"})
output

['天安门广场', '故宫', '颐和园']

In [ ]:
# LCEL： LangChain Execution Language（LangChain 表达语⾔）是⼀种声明性的⽅式来链接 LangChain 组件（工作流）。 

In [9]:
# 解析器作用在ChatPromptTemplate模版中

In [11]:
from langchain_openai import ChatOpenAI
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.prompts import ChatPromptTemplate

#构建提示词模版
prompt = ChatPromptTemplate.from_messages([
    ("system", "{parser_instructions}"),
    ("human", "列出{cityName}的{viewPointNum}个著名景点。")
])

#构建输出解析器并获取解析格式
output_parser = CommaSeparatedListOutputParser()
parser_instructions = output_parser.get_format_instructions()

#动态补充提示词内容
final_prompt = prompt.invoke({"cityName": "南京", "viewPointNum": 3, 
                              "parser_instructions": parser_instructions})

#最后，使用LangChain中的`chain`的抽象，合并最终的提示、大模型实例及OutPut Parse共同执行。
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
model = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")


response = model.invoke(final_prompt)
ret = output_parser.invoke(response)
print(ret)

['中山陵', '夫子庙', '玄武湖']


In [12]:
# **DatetimeOutputParser时间输出解析器示例**

In [13]:
from langchain.output_parsers import DatetimeOutputParser#日期输出解析器
from langchain.prompts import PromptTemplate

#制定输出解析器
output_parser = DatetimeOutputParser()

#制定提示词模版
template = """回答用户的问题：
{question}

{format_instructions}"""

#时间解析器的解析格式
format_instructions = output_parser.get_format_instructions()

#补充提示词模版
prompt = PromptTemplate.from_template(
    template,
    partial_variables={"format_instructions":format_instructions}
)

API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
model = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")

chain = prompt | model | output_parser
output = chain.invoke("周杰伦是什么时候出道的？")
output

datetime.datetime(2000, 11, 6, 0, 0)

In [14]:
# **EnumOutputParser枚举输出解析器示例**

In [19]:
from langchain.output_parsers.enum import EnumOutputParser
from enum import Enum

#定义枚举类型
class Colors(Enum):
    RED = "红色"
    BROWN = "棕色"
    BLACK = "黑色"
    WHITE = "白色"
    YELLOW = "黄色"
    
#制定输出解析器
parse = EnumOutputParser(enum=Colors)

#制定提示词模版
promptTemplate = PromptTemplate.from_template(
    """{person}的皮肤主要是什么颜色？
    
    {instructions}"""
)
#解析器的解析格式:原本解析器的英文解析格式会报错
# instructions = parse.get_format_instructions() 
instructions = "响应结果请选择以下选项之一：红色、棕色、黑色、白色和黄色。"
#提示词部分补充
prompt = promptTemplate.partial(instructions=instructions)

chain = prompt | model | parse
chain.invoke({"person":"亚洲人"})

<Colors.YELLOW: '黄色'>

In [21]:
# **注意：**直接使用输出解析器原始的英文的解析格式作用到提示词中可能由于中英文掺杂和中英文语义的区别导致模型报错，
# 因此，可以适当将输出解析器的解析格式手动翻译成英文后再用！

In [22]:
# **Pydantic JSON 输出解析器**

In [23]:
# JSON输出解析器允许用户指定任意JSON架构并查询LLMs以获取符合该框架的输出。

In [27]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel,Field
from langchain.prompts import PromptTemplate
from typing import List

#定义JSON结构
class Book(BaseModel):
    title:str = Field(description="书名")
    author:str = Field(description="作者")
    description:str = Field(description="书的简介")
    beLike:List[str] = Field(description="相关书籍的名称")
    
query = "请给我介绍下中国历史的经典书籍"

parser = JsonOutputParser(pydantic_object=Book)

format_instructions = parser.get_format_instructions()
# format_instructions = '''输出应格式化为符合以下JSON模式的JSON实例。JSON结构如下：{"title":"标题","author":"作者","description":"书的简介"}'''
prompt = PromptTemplate(
    template="{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions":format_instructions}
)

chain = prompt | model | parser
chain.invoke({"query":query})

{'title': '中国历史经典书籍介绍',
 'author': 'Various Authors',
 'description': '中国历史悠久，文化灿烂，留下了许多经典的历史书籍。这些书籍不仅记录了中国几千年的历史变迁，还蕴含了丰富的哲学思想和文化精髓。以下是一些中国历史的经典书籍，涵盖了从古代到近代的各个时期。',
 'beLike': ['《史记》',
  '《资治通鉴》',
  '《汉书》',
  '《三国志》',
  '《左传》',
  '《春秋》',
  '《明史》',
  '《清史稿》']}

In [28]:
# **xml输出解析器**

In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import XMLOutputParser

API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
model = ChatOpenAI(
    model_name="deepseek-chat",
    api_key=API_KEY,
    base_url="https://api.deepseek.com"
)

# 还有⼀个⽤于提示语⾔模型填充数据结构的查询意图。
actor_query = "⽣成周星驰的简化电影作品列表，按照最新的时间降序"

# 设置解析器 + 将指令注⼊提示模板。
parser = XMLOutputParser()
prompt = PromptTemplate(
    template="回答⽤户的查询。\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
# print(parser.get_format_instructions())

chain = prompt | model
response = chain.invoke({"query": actor_query})
xml_output = parser.parse(response.content)
print(response.content)

<?xml version="1.0" encoding="UTF-8"?>
<filmography>
  <film>
    <title>美人鱼</title>
    <year>2016</year>
    <role>导演/编剧/制片人</role>
  </film>
  <film>
    <title>西游·降魔篇</title>
    <year>2013</year>
    <role>导演/编剧/制片人</role>
  </film>
  <film>
    <title>长江7号</title>
    <year>2008</year>
    <role>导演/主演/编剧</role>
  </film>
  <film>
    <title>功夫</title>
    <year>2004</year>
    <role>导演/主演/编剧/制片人</role>
  </film>
  <film>
    <title>少林足球</title>
    <year>2001</year>
    <role>导演/主演/编剧</role>
  </film>
  <film>
    <title>喜剧之王</title>
    <year>1999</year>
    <role>导演/主演/编剧</role>
  </film>
</filmography>


In [30]:
xml_output

{'filmography': [{'film': [{'title': '美人鱼'},
    {'year': '2016'},
    {'role': '导演/编剧/制片人'}]},
  {'film': [{'title': '西游·降魔篇'}, {'year': '2013'}, {'role': '导演/编剧/制片人'}]},
  {'film': [{'title': '长江7号'}, {'year': '2008'}, {'role': '导演/主演/编剧'}]},
  {'film': [{'title': '功夫'}, {'year': '2004'}, {'role': '导演/主演/编剧/制片人'}]},
  {'film': [{'title': '少林足球'}, {'year': '2001'}, {'role': '导演/主演/编剧'}]},
  {'film': [{'title': '喜剧之王'}, {'year': '1999'}, {'role': '导演/主演/编剧'}]}]}

In [31]:
# **自定义输出解析器**

In [32]:
# 在某些情况下，我们可以实现自定义解析器以将模型输出内容构造成自定义的格式。

In [34]:
# from typing import Iterator
from langchain_core.messages import AIMessage,AIMessageChunk

#自定义输出解析器
def parse(ai_message:AIMessage)->str:
    #函数参数就是模型的输出。
    #swapcase表示将模型输出内容大小写进行相互转换后进行返回
    return ai_message.content.swapcase()

chain = model | parse
response = chain.invoke("are you ok?")
response

"i'M JUST A COMPUTER PROGRAM, SO i DON'T HAVE FEELINGS, BUT i'M FUNCTIONING PERFECTLY FINE! 😊 hOW ABOUT YOU? aRE *YOU* OKAY? iF YOU NEED SOMEONE TO TALK TO OR HAVE ANYTHING ON YOUR MIND, i'M HERE TO LISTEN OR HELP HOWEVER i CAN. 💙"